In [2]:
!pip install gtfparse
!pip install polars=='0.16.17'
!pip install pyarrow
!pip install anndata==0.8.0

In [ ]:
#Inputs: csv outputs of SoupX ambient RNA removal pipeline
#Ouputs: combined anndata object

In [1]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import pyarrow
from gtfparse import read_gtf
import scipy

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
#Input all csv outputs of SoupX
dat_list = ['../../SoupX/AC/Outputs/Run12_sample2_ncbi_soupcorrected.csv',
'../../SoupX/AC/Outputs/Run12_sample3_ncbi_soupcorrected.csv',
'../../SoupX/AC/Outputs/Run12_sample5_ncbi_soupcorrected.csv',
           '../../SoupX/AC/Outputs/Run12_sample6_ncbi_soupcorrected.csv',
           '../../SoupX/AC/Outputs/Run12_sample7_ncbi_soupcorrected.csv',
           '../../SoupX/AC/Outputs/Run12_sample9_ncbi_soupcorrected.csv',
           '../../SoupX/AC/Outputs/Run12_sample10_ncbi_soupcorrected.csv',
           '../../SoupX/AC/Outputs/Run12_sample11_ncbi_soupcorrected.csv',
           '../../SoupX/AC/Outputs/Run12_sample12_ncbi_soupcorrected.csv',
           '../../SoupX/AC/Outputs/Run17_sample2_ncbi_soupcorrected.csv',
           '../../SoupX/AC/Outputs/Run17_sample3_ncbi_soupcorrected.csv',
           '../../SoupX/AC/Outputs/Run17_sample4_ncbi_soupcorrected.csv',]

In [7]:
#Input the first sample here
tot_dat = ad.read_csv('../../SoupX/AC/Outputs/Run12_sample1_ncbi_soupcorrected.csv')

tot_dat = tot_dat.T
tot_dat.X = scipy.sparse.csr_matrix(tot_dat.X)
counts = np.sum(tot_dat.X, axis = 1).A.reshape((1,len(tot_dat)))[0]
genes = np.count_nonzero(tot_dat.X.A, axis = 1)

tot_dat.obs['n_counts'] = counts
tot_dat.obs['n_genes'] = genes
#manually set key name
tot_dat.obs['key'] = 'Run12_sample1'
tot_dat.var_names = [i for i in tot_dat.var_names]

for dn in dat_list:
    dat = ad.read_csv(dn)
    dat = dat.T
    dat.X = scipy.sparse.csr_matrix(dat.X)
    counts = np.sum(dat.X, axis = 1).A.reshape((1,len(dat)))[0]
    genes = np.count_nonzero(dat.X.A, axis = 1)
    
    dat.obs['n_counts'] = counts
    dat.obs['n_genes'] = genes
    #set this such that it outputs the key that you would like
    dat.obs['key'] = dn.split('_')[0].split('/')[-1] + '_' + dn.split('_')[1]
    dat.var_names = [i for i in dat.var_names]
    
    tot_dat = ad.concat([tot_dat, dat], join = 'outer')
    print(tot_dat.shape)

/scratch/miniconda/lib/python3.7/site-packages/anndata/_core/anndata.py:1828: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


(5796, 22251)
(8471, 22445)
(11061, 22633)
(13673, 22753)
(15714, 22796)
(18531, 22851)
(20895, 22902)
(24129, 22934)
(27043, 22957)
(33551, 23072)
(40169, 23172)
(50953, 23671)


In [8]:
tot_dat.var_names

Index(['LOC100337544', 'LOC100337546', 'LOC100551521', 'LOC100551532',
       'LOC100551533', 'LOC100551546', 'LOC100551547', 'LOC100551550',
       'LOC100551552', 'LOC100551560',
       ...
       'zswim7', 'zswim8', 'zswim9', 'zup1', 'zw10', 'zwilch', 'zxdc', 'zyx',
       'zzef1', 'zzz3'],
      dtype='object', length=23671)

In [10]:
#check to see if gene names match names in BLAST tables
mapping = pd.read_csv('../../BLASTMAPPING/maps/active_maps/hypo_proj/mgac/mg_to_ac.txt', delimiter = '\t', header = None)

In [13]:
a = 0
mo_set = set(mapping[1].unique())
for item in tot_dat.var_names:
    if item in mo_set:
        a += 1
a

18742

In [14]:
tot_dat.var_names

Index(['LOC100337544', 'LOC100337546', 'LOC100551521', 'LOC100551532',
       'LOC100551533', 'LOC100551546', 'LOC100551547', 'LOC100551550',
       'LOC100551552', 'LOC100551560',
       ...
       'zswim7', 'zswim8', 'zswim9', 'zup1', 'zw10', 'zwilch', 'zxdc', 'zyx',
       'zzef1', 'zzz3'],
      dtype='object', length=23671)

In [27]:
tot_dat.obs['key'].unique()

array(['Run12_sample1', 'Run12_sample2', 'Run12_sample3', 'Run12_sample5',
       'Run12_sample6', 'Run12_sample7', 'Run12_sample9',
       'Run12_sample10', 'Run12_sample11', 'Run12_sample12',
       'Run17_sample2', 'Run17_sample3', 'Run17_sample4'], dtype=object)

In [12]:
tot_dat.write('../../Testing_Raw_Dat_RNASEQ_Joined/tot_dat_AC_ncbi_soupx.h5ad')